# CO2 tax dose response in GREU (ModelFlow translation)

DREAM's CO2 tax shock with four sizes: `tCO2_Emarg` and `tCO2_xEmarg` + 250, 500, 750 or 1000 (the model's own
units), phased in as in DREAM's script (0 up to 2024, linear 2025-2030, full from 2030). Each size is solved on
two versions of the model:

* **CGE**: the energy technology block switched off (DREAM's S2 set-up), solved from the baseline.
* **Technology**: the energy technology block on (DREAM's S3 set-up), solved from DREAM's staged start
  (energy prices, then the technology shares, then the whole model; `tools/et_start.py`).

The +750 runs are DREAM's own S2 and S3; ModelFlow reproduces them (max relative difference 1e-5 and 6e-7).

**Reference.** DREAM's S2 script also switches the energy demand by purpose (`qEpj`) from fixed coefficients to a
price-elastic form. That switch alone moves the CGE economy (notebook `greu_s2_switch.ipynb`). The CGE runs are
therefore measured against a zero-tax run on the same model, so only the tax is in the numbers. The technology
runs are measured against the baseline, which that model reproduces exactly.

The runs are made by `python tools/own_shocks.py dose both` (about 25 minutes); this notebook only reads them.

*Example calculations based on example data. The model is GREU (ModelFlow translation), not DREAM's GreenREFORM-EU.*

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'tools'))
import greu_viz as gv
pd.set_option('display.width', 200)

In [2]:
LEVELS = [250, 500, 750, 1000]
runs = {('cge', L): gv.solution(f'own_dose_cge_{L}') for L in LEVELS}
runs.update({('et', L): gv.solution(f'own_dose_et_{L}') for L in LEVELS})
ref = {'cge': gv.solution('own_dose_cge_0'), 'et': gv.baseline('main_energy_technology')}
MODEL = {'cge': 'CGE (technology block off)', 'et': 'With the technology block'}
YEARS = list(range(2021, 2051))
for df in list(runs.values()) + list(ref.values()):
    df['ENERGY_PJ'] = gv.energy_pj(df)      # physical energy use, PJ (qE is the CGE's real energy input)

def effect(var, model, level, years=YEARS):
    """% difference from the reference run"""
    return gv.pct(runs[(model, level)].loc[years, var], ref[model].loc[years, var])

## Emissions over time

Total greenhouse gas emissions (`qEmmTot[co2e,unfccc]`). The tax is phased in over 2025-2030; after that the
effect is close to constant. Darker lines are higher taxes.

In [3]:
V = 'QEMMTOT__CO2E__UNFCCC'
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, subplot_titles=[MODEL['cge'], MODEL['et']],
                    horizontal_spacing=0.04)
for j, model in enumerate(['cge', 'et'], 1):
    for k, L in enumerate(LEVELS):
        y = effect(V, model, L)
        fig.add_trace(go.Scatter(x=YEARS, y=y, mode='lines', line=dict(color=gv.ORD4[k], width=2),
                                 name=f'+{L}', legendgroup=str(L), showlegend=(j == 1),
                                 hovertemplate=f'+{L}, %{{x}}: %{{y:.2f}} %<extra></extra>'), row=1, col=j)
        if j == 2:      # direct label at the end of the line
            fig.add_annotation(x=2050, y=y.iloc[-1], text=f'+{L}', showarrow=False, xanchor='left', xshift=4,
                               font=dict(size=11, color='#52514e'), row=1, col=j)
fig.update_layout(template=gv.TEMPLATE, height=420, hovermode='x unified', legend_title_text='CO2 tax',
                  title='Greenhouse gas emissions, % difference from the reference')
fig.update_yaxes(title_text='%', row=1, col=1, zeroline=True, zerolinecolor='#d6d5d0')
fig.show()

## How much each step of the tax removes

The abatement in 2030 against the size of the tax, and what each extra 250 adds. The two versions of the model
lie almost on top of each other: the technology block covers only two energy services (heating in the dairy
industry, processes under the ETS in cement), so at the level of the whole economy the abatement comes from the
substitution in the CGE part.

In [4]:
Y = 2030
ab = pd.DataFrame({m: [-effect(V, m, L, [Y]).iloc[0] for L in LEVELS] for m in ['cge', 'et']}, index=LEVELS)
fig = make_subplots(rows=1, cols=2, subplot_titles=['Abatement in 2030, % of emissions',
                                                    'Abatement of each extra 250, percentage points'],
                    horizontal_spacing=0.1)
for i, m in enumerate(['cge', 'et']):
    fig.add_trace(go.Scatter(x=LEVELS, y=ab[m], mode='lines+markers', name=MODEL[m],
                             line=dict(color=gv.CAT[i], width=2), marker=dict(size=9),
                             hovertemplate='+%{x}: %{y:.2f} %<extra>' + MODEL[m] + '</extra>'), row=1, col=1)
    step = ab[m].diff().fillna(ab[m])
    fig.add_trace(go.Bar(x=[f'{L - 250}→{L}' for L in LEVELS], y=step, name=MODEL[m], showlegend=False,
                         marker=dict(color=gv.CAT[i], line=dict(width=2, color='#fcfcfb')),
                         hovertemplate='%{x}: %{y:.2f} pp<extra>' + MODEL[m] + '</extra>'), row=1, col=2)
fig.update_layout(template=gv.TEMPLATE, height=400, barmode='group', bargap=0.35)
fig.update_xaxes(title_text='CO2 tax', row=1, col=1)
fig.update_yaxes(rangemode='tozero')
fig.show()
ab.round(2)

,cge,et
250,2.11,2.18
500,3.98,4.03
750,5.66,5.67
1000,7.18,7.16


## Cost against abatement

Real GDP loss against the emission reduction in 2030, one point per tax level. The cost per unit of
abatement rises with the tax, since each step abates less while the GDP loss grows about linearly.

In [5]:
fig = go.Figure()
for i, m in enumerate(['cge', 'et']):
    x = ab[m]
    y = [-effect('QGDP', m, L, [Y]).iloc[0] for L in LEVELS]
    fig.add_trace(go.Scatter(x=x, y=y, mode='lines+markers+text', name=MODEL[m], text=[f'+{L}' for L in LEVELS],
                             textposition='top left' if m == 'cge' else 'bottom right',
                             textfont=dict(size=11, color='#52514e'),
                             line=dict(color=gv.CAT[i], width=2), marker=dict(size=9),
                             hovertemplate='abatement %{x:.2f} %, GDP loss %{y:.3f} %<extra>' + MODEL[m] + '</extra>'))
fig.update_layout(template=gv.TEMPLATE, height=420, title='GDP loss and abatement in 2030 (%)',
                  xaxis_title='emission reduction, %', yaxis_title='real GDP loss, %')
fig.update_xaxes(rangemode='tozero'); fig.update_yaxes(rangemode='tozero')
fig.show()

## Who abates

Emissions by tax group in 2030 at +750 (DREAM's own shock size). Households' energy and companies' energy
fall by about 10 %, the non-energy emissions (agriculture, processes) by 2 %.

In [6]:
groups = {'QCO2E_TAXGROUP__ENERGY_CORP': 'Energy, companies', 'QCO2E_TAXGROUP__ENERGY_HH': 'Energy, households',
          'QCO2E_TAXGROUP__NON_ENERGY': 'Non-energy'}
fig = go.Figure()
for i, m in enumerate(['cge', 'et']):
    vals = [effect(v, m, 750, [Y]).iloc[0] for v in groups]
    fig.add_trace(go.Bar(y=list(groups.values()), x=vals, orientation='h', name=MODEL[m],
                         marker=dict(color=gv.CAT[i], line=dict(width=2, color='#fcfcfb')),
                         text=[f'{v:.1f} %' for v in vals], textposition='outside',
                         textfont=dict(color='#52514e'),
                         hovertemplate='%{y}: %{x:.2f} %<extra>' + MODEL[m] + '</extra>'))
fig.update_layout(template=gv.TEMPLATE, height=330, barmode='group', bargap=0.3,
                  title='CO2e by tax group, 2030, CO2 tax +750, % difference', xaxis_title='%')
fig.update_yaxes(autorange='reversed')
fig.show()

## The numbers

The headline variables in 2030 and 2050 (`VLUMPSUM`, the lump sum that balances the public budget, as an
absolute difference; everything else in %).

In [7]:
show = {'QGDP': 'Real GDP', 'QC': 'Private consumption', 'QI': 'Investment', 'QX': 'Exports', 'QM': 'Imports',
        'ENERGY_PJ': 'Energy use, PJ', 'QE': 'Real energy inputs (CGE volume)', 'PGDP': 'GDP deflator', 'QEMMTOT__CO2E__UNFCCC': 'CO2e emissions',
        'VLUMPSUM': 'Lump sum (abs.)'}
rows = {}
for m in ['cge', 'et']:
    for L in LEVELS:
        for y in (2030, 2050):
            r, b = runs[(m, L)], ref[m]
            rows[(MODEL[m], L, y)] = {lab: (r.at[y, v] - b.at[y, v]) if v == 'VLUMPSUM' else gv.pct(r.at[y, v], b.at[y, v])
                                      for v, lab in show.items()}
pd.DataFrame(rows).round(3)

CGE (technology block off)                                                   With the technology block                                                  
                                                      250           500           750            1000                             250           500           750            1000       
                                                      2030   2050   2030   2050   2030   2050    2030   2050                      2030   2050   2030   2050   2030   2050    2030   2050
Real GDP                                            -0.092 -0.132 -0.185 -0.260 -0.277 -0.385  -0.368 -0.506                    -0.088 -0.124 -0.176 -0.245 -0.263 -0.362  -0.350 -0.476
Private consumption                                 -0.040 -0.140 -0.087 -0.279 -0.140 -0.418  -0.198 -0.554                    -0.040 -0.134 -0.088 -0.268 -0.140 -0.400  -0.197 -0.530
Investment                                          -0.289 -0.136 -0.560 -0.265 -0.815 -0.388  -1.054 -0.504                    -0.273 -0.127 -0.527 -0.247 -0.767 -0.362  -0.993 -0.472
Exports                                             -0.199 -0.268 -0.383 -0.516 -0.555 -0.747  -0.717 -0.963                    -0.189 -0.256 -0.364 -0.492 -0.528 -0.711  -0.683 -0.918
Imports                                             -0.202 -0.206 -0.387 -0.395 -0.558 -0.570  -0.718 -0.734                    -0.194 -0.197 -0.373 -0.377 -0.538 -0.545  -0.692 -0.701
Energy use, PJ                                      -0.655 -0.692 -1.237 -1.308 -1.763 -1.865  -2.241 -2.372                    -0.646 -0.684 -1.224 -1.296 -1.747 -1.851  -2.225 -2.359
Real energy inputs (CGE volume)                     -1.400 -1.442 -2.633 -2.714 -3.732 -3.848  -4.721 -4.870                    -1.407 -1.452 -2.644 -2.731 -3.749 -3.874  -4.746 -4.906
GDP deflator                                         0.150  0.186  0.292  0.360  0.426  0.524   0.554  0.678                     0.148  0.182  0.288  0.353  0.420  0.513   0.546  0.666
CO2e emissions                                      -2.109 -2.183 -3.980 -4.122 -5.660 -5.866  -7.182 -7.447                    -2.177 -2.263 -4.035 -4.197 -5.674 -5.906  -7.161 -7.458
Lump sum (abs.)                                      3.067  2.657  5.849  5.061  8.391  7.256  10.728  9.272                     3.039  2.673  5.796  5.094  8.328  7.316  10.664  9.365